# Register the Agent in the Registry (Optional)

::: This notebook is **optional**. It demonstrates how to register the deployed travel agent
as a discoverable A2A (Agent-to-Agent) service in the Module 3b Registry. Skip if you only
want to run the agent. :::

By registering the agent, the Registry becomes a single catalog of both **tools** (MCP) and
**agents** (A2A). Other agents can discover the travel planner through the Registry API.

Steps:

1. Read `RuntimeArn` from FAST-stack outputs and build the agent URL
2. Find the Registry created in Module 3b
3. Assume the publisher role and create an A2A registry record
4. Assume the admin role and approve it
5. Drop back to the default identity and list records to verify


## Step 1: Build the Agent URL


In [ ]:
import os
import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError(
        "No AWS region configured. Set AWS_REGION (or AWS_DEFAULT_REGION), "
        "or run `aws configure set region <region>`."
    )
cfn = boto3.client("cloudformation", region_name=REGION)

outputs = {
    o["OutputKey"]: o["OutputValue"]
    for o in cfn.describe_stacks(StackName="FAST-stack")["Stacks"][0]["Outputs"]
}
RUNTIME_ARN = outputs["RuntimeArn"]
RUNTIME_ID = RUNTIME_ARN.rsplit("/", 1)[-1]
AGENT_URL = f"https://{RUNTIME_ID}.runtime.bedrock-agentcore.{REGION}.amazonaws.com"

print(f"Runtime ARN: {RUNTIME_ARN}")
print(f"Agent URL:   {AGENT_URL}")

## Step 2: Find the Registry

The Registry was created interactively in Module 3b. There is no CloudFormation export — list
registries and take the first one.


In [ ]:
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

registries = agentcore.list_registries().get("registries", [])
if not registries:
    raise RuntimeError("No registry found. Complete Module 3b Step 3 first.")

# Prefer the canonical workshop-registry; fall back to the first only if it is the sole choice.
workshop_match = [r for r in registries if r.get("name") == "workshop-registry"]
if workshop_match:
    REGISTRY_ID = workshop_match[0]["registryId"]
elif len(registries) == 1:
    REGISTRY_ID = registries[0]["registryId"]
else:
    names = [r.get("name") for r in registries]
    raise RuntimeError(
        f"Multiple registries exist ({names}) but none is named 'workshop-registry'. "
        f"Disambiguate manually by setting REGISTRY_ID."
    )
print(f"Registry ID: {REGISTRY_ID}")

## Step 3: Assume the Publisher Role

The Registry uses role-based access control — publishers create records, admins approve them.


In [ ]:
sts = boto3.client("sts", region_name=REGION)

exports = {}
for page in cfn.get_paginator("list_exports").paginate():
    for e in page["Exports"]:
        exports[e["Name"]] = e["Value"]

PUBLISHER_ROLE_ARN = exports["ac-RegistryPublisherRoleArn"]
ADMIN_ROLE_ARN = exports["ac-RegistryAdminRoleArn"]


def _client_as(role_arn, service, session_name):
    creds = sts.assume_role(RoleArn=role_arn, RoleSessionName=session_name)["Credentials"]
    return boto3.client(
        service,
        region_name=REGION,
        aws_access_key_id=creds["AccessKeyId"],
        aws_secret_access_key=creds["SecretAccessKey"],
        aws_session_token=creds["SessionToken"],
    )


publisher_agentcore = _client_as(PUBLISHER_ROLE_ARN, "bedrock-agentcore-control", "publisher")
print(f"Assumed publisher role: {PUBLISHER_ROLE_ARN}")


## Step 4: Create the A2A Record

Build the A2A agent card as a Python dict, embed it as the `inlineContent` of the descriptor,
and submit.


In [ ]:
import json

agent_card = {
    "protocolVersion": "0.3",
    "name": "Travel Planning Agent",
    "description": "Plans trips by searching flights, hotels, and knowledge bases. Deployed on AgentCore Runtime via FAST.",
    "url": AGENT_URL,
    "version": "1.0.0",
    "capabilities": {"streaming": False, "pushNotifications": False},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": [
        {"id": "plan_trip",      "name": "Plan Trip",      "description": "Plans complete trips including flights and hotels", "tags": ["travel", "planning"]},
        {"id": "search_flights", "name": "Search Flights", "description": "Searches available flights between cities",         "tags": ["travel", "flights"]},
        {"id": "search_hotels",  "name": "Search Hotels",  "description": "Searches available hotels in a destination",        "tags": ["travel", "hotels"]},
    ],
}
descriptors = {
    "a2a": {
        "agentCard": {
            "schemaVersion": "0.3",
            "inlineContent": json.dumps(agent_card),
        }
    }
}

RECORD_NAME = "workshop_travel_agent"

try:
    publisher_agentcore.create_registry_record(
        registryId=REGISTRY_ID,
        name=RECORD_NAME,
        descriptorType="A2A",
        descriptors=descriptors,
        recordVersion="1.0",
    )
    print(f"Created record '{RECORD_NAME}'")
except publisher_agentcore.exceptions.ConflictException:
    print(f"Record '{RECORD_NAME}' already exists — continuing")

# The recordId is not returned by CreateRegistryRecord (only recordArn + status),
# so look it up by name from the list.
existing = publisher_agentcore.list_registry_records(registryId=REGISTRY_ID).get("registryRecords", [])
RECORD_ID = next(r["recordId"] for r in existing if r["name"] == RECORD_NAME)
print(f"Record ID: {RECORD_ID}")


## Step 5: Submit and Approve the Record

The Registry state machine is `DRAFT → PENDING_APPROVAL → APPROVED`. The publisher submits
the record, then the admin approves it (separation of duties). This cell waits for the
record to leave `CREATING`, submits as publisher, then approves as admin. It is idempotent
— if the record is already `APPROVED` it does nothing.


In [ ]:
import time

admin_agentcore = _client_as(ADMIN_ROLE_ARN, "bedrock-agentcore-control", "admin")


def _wait_until_not(client, status_to_leave, poll_seconds=5, attempts=12):
    """Poll get_registry_record until status != status_to_leave (e.g. CREATING/UPDATING)."""
    for i in range(attempts):
        rec = client.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_ID)
        st = rec.get("status", "UNKNOWN")
        if st != status_to_leave:
            return st
        print(f"  [{(i + 1) * poll_seconds}s] still {status_to_leave}...")
        time.sleep(poll_seconds)
    return rec.get("status", "UNKNOWN")


# The Registry state machine requires: DRAFT -> PENDING_APPROVAL -> APPROVED.
# CreateRegistryRecord returns CREATING, which settles to DRAFT once the record
# is persisted. The publisher must submit before the admin can approve.
status = _wait_until_not(publisher_agentcore, "CREATING")
print(f"Record {RECORD_ID} current status: {status}")

if status == "DRAFT":
    publisher_agentcore.submit_registry_record_for_approval(
        registryId=REGISTRY_ID,
        recordId=RECORD_ID,
    )
    print(f"Submitted {RECORD_ID} -> PENDING_APPROVAL")
    status = _wait_until_not(publisher_agentcore, "DRAFT", attempts=6)

if status == "APPROVED":
    print(f"Record {RECORD_ID} already APPROVED - nothing to do")
elif status == "PENDING_APPROVAL":
    admin_agentcore.update_registry_record_status(
        registryId=REGISTRY_ID,
        recordId=RECORD_ID,
        status="APPROVED",
        statusReason="Approved for workshop",
    )
    print(f"Record {RECORD_ID} approved")
else:
    raise RuntimeError(
        f"Cannot approve record {RECORD_ID} from status {status!r}. "
        f"Expected DRAFT/PENDING_APPROVAL/APPROVED."
    )


## Step 6: Verify

Fall back to the default identity and list the registry records — the travel agent should now
show up with `status=APPROVED`.


In [ ]:
records = agentcore.list_registry_records(registryId=REGISTRY_ID).get("registryRecords", [])
for rec in records:
    marker = ">" if rec["name"] == "workshop_travel_agent" else " "
    print(f"{marker} {rec['name']:30s} type={rec['descriptorType']:6s} status={rec.get('status', '?')}")


## What This Means

| Step | Who | What |
|------|-----|------|
| Register tools | Platform team (Module 3b) | MCP tools cataloged in the Registry |
| Expose tools | Platform team (Module 3b) | AgentCore Gateway serves tools over MCP |
| Build agent | AI/ML engineer (Module 4) | Agent consumes tools and models |
| Deploy agent | AI/ML engineer (Module 4) | Agent runs on AgentCore Runtime |
| **Register agent** | **AI/ML engineer (Module 4)** | **Agent discoverable as A2A service** |

The Registry is now a complete catalog of both **tools** (MCP) and **agents** (A2A), which
unlocks A2A discovery and routing across the platform.

Proceed to **notebook 08 — Cleanup**.
